# AiJockey — Tier 1.5: CLAP DJ-Compatibility Head

Transfer learning on top of frozen CLAP. Trains MLP projection head via
InfoNCE contrastive loss. Result: cosine similarity in projected space
reflects DJ-mixability instead of generic acoustic similarity.

**Time on T4**: triplet build ~1 min, train ~3-5 min for 50 epochs.

**Inputs**: pre-analyzed clips in `cache/` (already done if you ran analyze).

**Output**: `checkpoints/clap_compat_head.pt`. Use via `--compat_head` flag
on `main.py plan`.

## 1. Setup

In [ ]:
!nvidia-smi | head -10
!apt-get install -y rubberband-cli ffmpeg > /dev/null 2>&1
!git clone https://github.com/architagrawal/aiJockey.git 2>/dev/null || (cd aiJockey && git pull)
%cd aiJockey

In [ ]:
# Minimal deps for THIS notebook (no audio decoding needed — uses cached CLAP only)
!pip install -q "numpy<2.0"
!pip install -q torch matplotlib

## 2. Verify analyzed clips exist

Need cache/<id>.json + cache/<id>.npz from a prior analyze run.
If you don't have one, run analyze in the pipeline notebook first.

In [ ]:
import os
n = len([f for f in os.listdir('cache') if f.endswith('.json')]) if os.path.exists('cache') else 0
print(f'analyzed clips in cache/: {n}')
if n < 3:
    print('NEED >=3 ANALYZED CLIPS. Run pipeline notebook analyze step first.')

## 3. Build triplet dataset

For each anchor clip: find compatible positive (similar BPM + Camelot wheel
neighbor) and K incompatible negatives (large BPM gap or far Camelot dist).

If clip pool small (<50), augments with synthetic perturbations of CLAP
vectors so you still get usable training signal.

In [ ]:
!python src/training/clap_pairs.py \
    --cache cache/ \
    --out datasets/clap_triplets.npz \
    --n_anchors 500 \
    --n_neg 8

In [ ]:
import numpy as np
d = np.load('datasets/clap_triplets.npz')
print('anchor:   ', d['anchor'].shape)
print('positive: ', d['positive'].shape)
print('negatives:', d['negatives'].shape)

## 4. Train compat head

In [ ]:
!python src/training/clap_finetune.py \
    --triplets datasets/clap_triplets.npz \
    --ckpt checkpoints/clap_compat_head.pt \
    --epochs 50 \
    --batch_size 64 \
    --lr 1e-3

## 5. Plot training history

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open('checkpoints/clap_compat_head.history.json'))
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(h['train_loss'], label='train')
ax[0].plot(h['val_loss'], label='val')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('InfoNCE loss'); ax[0].legend()
ax[1].plot(h['val_acc'])
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('val acc (positive > all negatives)')
plt.tight_layout(); plt.show()

## 6. Sanity check — projected similarity matrix

Should show high cosine for compatible clip pairs, low for incompatible.

In [ ]:
import sys; sys.path.insert(0, 'src')
import sys; sys.path.insert(0, 'src/training')
import numpy as np
import json
from pathlib import Path
from clap_finetune import load_compat_head, project_batch

head = load_compat_head('checkpoints/clap_compat_head.pt')

# Load cached clips
clips = []
for jp in sorted(Path('cache').glob('*.json')):
    meta = json.load(open(jp))
    npz = np.load(str(Path('cache') / f'{jp.stem}.npz'))
    clips.append({
        'id': jp.stem,
        'tempo': meta.get('tempo', 0),
        'key': meta.get('key', '?'),
        'clap': npz['clap'].astype(np.float32),
    })
claps = np.stack([c['clap'] for c in clips])
compat = project_batch(head, claps)
compat_norm = compat / (np.linalg.norm(compat, axis=1, keepdims=True) + 1e-8)
raw_norm = claps / (np.linalg.norm(claps, axis=1, keepdims=True) + 1e-8)
sim_compat = compat_norm @ compat_norm.T
sim_raw = raw_norm @ raw_norm.T

print('Pairwise cosine — RAW CLAP vs COMPAT:')
for i, ci in enumerate(clips):
    for j, cj in enumerate(clips):
        if i >= j: continue
        bpm_diff = abs(ci['tempo'] - cj['tempo'])
        print(f"  {ci['id'][:30]:30s} <-> {cj['id'][:30]:30s} "
              f"bpm_diff={bpm_diff:5.1f}  raw={sim_raw[i,j]:.3f}  compat={sim_compat[i,j]:.3f}")

## 7. Generate mix using compat head + classifier

Stack everything: trained classifier picks technique + compat head improves
candidate scoring.

In [ ]:
# Skip if no clips available — only re-run if pipeline already set up
import os
if os.path.exists('clips') and any(
    f.endswith(('.wav','.mp3','.flac')) for f in os.listdir('clips')):
    !python src/main.py plan \
        --cache cache/ \
        --duration 300 \
        --compat_head checkpoints/clap_compat_head.pt \
        --classifier checkpoints/technique_classifier.pt 2>/dev/null || \
    python src/main.py plan \
        --cache cache/ \
        --duration 300 \
        --compat_head checkpoints/clap_compat_head.pt
else:
    print('No clips. Skip — run pipeline notebook to generate full mix.')

## 8. Save head to Drive (persist between sessions)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil, os
# os.makedirs('/content/drive/MyDrive/AiJockey_models', exist_ok=True)
# shutil.copy('checkpoints/clap_compat_head.pt', '/content/drive/MyDrive/AiJockey_models/')
# print('saved to Drive')

## What this gives us

- **Smarter scoring**: planner's `timbre_score` now uses DJ-compat space.
  Two house tracks at compatible BPMs/keys score higher than two random
  EDM tracks even if their raw CLAP is similar.
- **No raw audio needed at training time** — uses pre-extracted CLAP
  embeddings. Tiny compute.
- **No HF fine-tune** — frozen CLAP, only the small projection head trains.
  Your model. AGPL-3.0. Tiny weights (~80KB).
- **Stacks with technique classifier** — both trained models active via
  --classifier and --compat_head flags.

**Limitation**: synthetic positive/negative labels (heuristic, not real).
Replace with Tier 2 real DJ-mix data for production.

**Hackathon talking point**: "transfer learning chain — frozen CLAP +
frozen Demucs + custom contrastive head trained on DJ-compatibility task."